In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Read CSV files
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Convert timestamps to timezone-aware datetimes (Asia/Shanghai)
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Components list: per-file counts of cmdb_id
metric_counts = metric_df['cmdb_id'].value_counts(dropna=False).rename_axis('cmdb_id').reset_index(name='count')
metric_counts['file'] = 'metric'
trace_counts = trace_df['cmdb_id'].value_counts(dropna=False).rename_axis('cmdb_id').reset_index(name='count')
trace_counts['file'] = 'trace'
log_counts = log_df['cmdb_id'].value_counts(dropna=False).rename_axis('cmdb_id').reset_index(name='count')
log_counts['file'] = 'log'
error_counts = error_logs_df['cmdb_id'].value_counts(dropna=False).rename_axis('cmdb_id').reset_index(name='count')
error_counts['file'] = 'error_logs'

components_df = pd.concat([metric_counts, trace_counts, log_counts, error_counts], ignore_index=True)
components_df = components_df[['file', 'cmdb_id', 'count']].sort_values(['file','count'], ascending=[True, False]).head(50)

# Convert to CSV-like text
components_csv = components_df.to_csv(index=False)

# 2) KPI/feature lists with count, p05, p50, p95 (use full series)
def summarise_percentiles(df, group_cols, value_col='value', file_label=''):
    if df.shape[0] == 0:
        return pd.DataFrame(columns=['file'] + list(group_cols) + ['count','p05','p50','p95'])
    agg = df.groupby(list(group_cols))[value_col].agg(
        count='count',
        p05=lambda x: float(np.nanpercentile(x.values, 5)) if x.notna().any() else np.nan,
        p50=lambda x: float(np.nanpercentile(x.values, 50)) if x.notna().any() else np.nan,
        p95=lambda x: float(np.nanpercentile(x.values, 95)) if x.notna().any() else np.nan
    ).reset_index()
    # Round percentiles for compactness
    for col in ['p05','p50','p95']:
        agg[col] = agg[col].round(6)
    agg.insert(0, 'file', file_label)
    return agg

metric_feats = summarise_percentiles(metric_df, ['cmdb_id','kpi_name'], 'value', 'metric')
trace_feats = summarise_percentiles(trace_df, ['cmdb_id','trace_name'], 'value', 'trace')
log_feats = summarise_percentiles(log_df, ['cmdb_id','log_name'], 'value', 'log')

# Limit to top 50 rows each to keep output compact
metric_feats_limited = metric_feats.sort_values(['cmdb_id','kpi_name']).head(50)
trace_feats_limited = trace_feats.sort_values(['cmdb_id','trace_name']).head(50)
log_feats_limited = log_feats.sort_values(['cmdb_id','log_name']).head(50)

metric_csv = metric_feats_limited.to_csv(index=False)
trace_csv = trace_feats_limited.to_csv(index=False)
log_csv = log_feats_limited.to_csv(index=False)

# 3) error_logs.csv: up to 20 earliest rows sorted by timestamp with tz handling
if 'ts' in error_logs_df.columns:
    err_sorted = error_logs_df.sort_values('ts').head(20).copy()
    # Format timestamp as ISO-like string with timezone offset
    err_sorted['timestamp_iso'] = err_sorted['ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    error_logs_out = err_sorted[['timestamp_iso','cmdb_id','message']].rename(columns={'timestamp_iso':'timestamp'})
else:
    err_sorted = error_logs_df.sort_values('timestamp').head(20).copy()
    # fallback: convert epoch to tz-aware
    err_sorted['ts'] = pd.to_datetime(err_sorted['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    err_sorted['timestamp'] = err_sorted['ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    error_logs_out = err_sorted[['timestamp','cmdb_id','message']]

error_logs_csv = error_logs_out.to_csv(index=False)

# Display compact CSV-like plain text outputs (each variable is a CSV string)
components_csv, metric_csv, trace_csv, log_csv, error_logs_csv

```
Out[1]:
```
summary = (
    "Summary of telemetry files:\n\n"
    "1) Components (per-file counts, top entries):\n"
    "- metric: top components by record count include Mysql01 (4092), Mysql02 (4074), Redis01 (2255), Redis02 (2229), Tomcat04 (1620), Tomcat02 (1591), Tomcat03 (1573), IG01 (1466), IG02 (1435), Tomcat01 (1417).\n"
    "- trace: top components include MG02 (1182), MG01 (1182), Tomcat04 (459), Tomcat02 (459), dockerB1 (459), dockerA2 (459), Tomcat03 (456), Tomcat01 (456), IG01 (453), IG02 (453).\n"
    "- log: most records are from Tomcat02 (92), Tomcat01 (86), Tomcat03 (86), Tomcat04 (84), apache02 (62), apache01 (62), Mysql01 (42).\n"
    "- error_logs: contains at least one entry for Tomcat04 (earliest shown).\n\n"
    "2) Metric KPI examples (medians and percentiles from full series):\n"
    "- IG01 JVM memory: HeapMemoryUsage median ~22.9956, HeapMemoryUsed median ~478,570,128, HeapMemoryMax constant 2,040,135,680.\n"
    "- IG01 JVM CPU/load: JVM_CPULoad median ~0.25585; CPU user-time median ~25.21; overall CPU util median ~25.8502.\n"
    "- Filesystem/disk: IG01 'tomcat' FSUsedSpace median ~12,572 MB; many filesystem metrics are constant or near-constant across samples.\n\n"
    "3) Trace feature examples (medians from full series):\n"
    "- IG01 trace.self.duration_mean median ~6.109118 (p95 ~6.8439); related per-destination traces (to Tomcat01/02/03/04) show similar median durations in the ~5.9–6.2s range.\n"
    "- IG02, MG01 traces show similar patterns with medians around ~6s for self.duration_mean and higher p95 values.\n\n"
    "4) Log feature examples (medians from full series):\n"
    "- Mysql01: log.error_count median = 0, log.row_count median = 2.0.\n"
    "- Tomcat instances: log.row_count medians ranging ~199–216; gc_pause_max medians ~0.06–0.07s; log.error_count medians = 0.\n"
    "\n"
    "5) error_logs sample (earliest shown):\n"
    "- 2021-03-10T17:59:24+0800, Tomcat04: an HTTP 500 error for IG01 POST /UOCP/person/ServiceTest9.json (message truncated in log).\n\n"
    "Overall: metric and trace datasets show heavy activity on database (Mysql), cache (Redis), gateway/ingress (IG), and Tomcat application nodes. Traces indicate backend call durations centered around ~6s for IG and MG paths. Logs show many request/row_count samples with almost zero error_count medians, but at least one 500 error is present in error_logs for Tomcat04 which may warrant investigation."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

('file,cmdb_id,count\nerror_logs,Tomcat04,1\nlog,Tomcat02,92\nlog,Tomcat01,86\nlog,Tomcat03,86\nlog,Tomcat04,84\nlog,apache02,62\nlog,apache01,62\nlog,Mysql01,42\nlog,Mysql02,2\nmetric,Mysql01,4092\nmetric,Mysql02,4074\nmetric,Redis01,2255\nmetric,Redis02,2229\nmetric,Tomcat04,1620\nmetric,Tomcat02,1591\nmetric,Tomcat03,1573\nmetric,IG01,1466\nmetric,IG02,1435\nmetric,Tomcat01,1417\nmetric,apache01,1308\nmetric,apache02,1269\nmetric,MG01,1174\nmetric,MG02,1166\nmetric,ServiceTest3,124\nmetric,ServiceTest11,124\nmetric,ServiceTest10,124\nmetric,ServiceTest1,124\nmetric,ServiceTest2,124\nmetric,ServiceTest4,124\nmetric,ServiceTest5,124\nmetric,ServiceTest8,124\nmetric,ServiceTest9,124\nmetric,ServiceTest6,124\nmetric,ServiceTest7,124\nmetric,dockerA2,51\nmetric,dockerB2,32\nmetric,dockerA1,29\nmetric,dockerB1,17\ntrace,MG02,1182\ntrace,MG01,1182\ntrace,Tomcat04,459\ntrace,Tomcat02,459\ntrace,dockerB1,459\ntrace,dockerA2,459\ntrace,Tomcat03,456\ntrace,Tomcat01,456\ntrace,IG01,453\ntrace,IG02,453\ntrace,dockerB2,453\ntrace,dockerA1,450\n', 'file,cmdb_id,kpi_name,count,p05,p50,p95\nmetric,IG01,JVM-Memory_7778_JVM_Memory_HeapMemoryMax,31,2040135680.0,2040135680.0,2040135680.0\nmetric,IG01,JVM-Memory_7778_JVM_Memory_HeapMemoryUsage,20,6.283475,22.9956,38.825065\nmetric,IG01,JVM-Memory_7778_JVM_Memory_HeapMemoryUsed,31,106814508.0,478570128.0,842213152.0\nmetric,IG01,JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed,31,57925184.0,57925184.0,57925184.0\nmetric,IG01,JVM-Operating System_7778_JVM_JVM_CPULoad,20,0.220125,0.25585,0.41252\nmetric,IG01,JVM-Runtime_7778_JVM_JVM_Uptime,21,447704.0,448544.0,449325.0\nmetric,IG01,JVM-Threads_7778_JVM_ThreadCount_Threads,31,67.0,71.0,75.0\nmetric,IG01,OSLinux-CPU_CPU-0_SingleCpuUtil,11,0.78185,1.0476,15.6558\nmetric,IG01,OSLinux-CPU_CPU-0_SingleCpuidle,15,89.148951,98.9192,99.204764\nmetric,IG01,OSLinux-CPU_CPU-2_SingleCpuUtil,11,0.84835,1.0484,2.32955\nmetric,IG01,OSLinux-CPU_CPU-2_SingleCpuidle,15,97.22867,98.951573,99.144883\nmetric,IG01,OSLinux-CPU_CPU_CPUCpuUtil,21,25.6953,25.8502,26.1153\nmetric,IG01,OSLinux-CPU_CPU_CPULoad,31,1.0,1.03,1.135\nmetric,IG01,OSLinux-CPU_CPU_CPUSysTime,31,0.291,0.353416,0.430253\nmetric,IG01,OSLinux-CPU_CPU_CPUUserTime,31,25.1653,25.21,25.319608\nmetric,IG01,OSLinux-CPU_CPU_CPUWio,31,0.0166,0.06652,0.4802\nmetric,IG01,OSLinux-CPU_CPU_CPUidleutil,31,73.678528,74.18,74.308342\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace,10,9951.0,9951.0,9951.0\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-_FSCapacity,15,30.2645,30.264562,30.2646\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent,15,16.366374,16.3664,16.3664\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-_FSUsedSpace,10,4319.0,4319.0,4319.0\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-boot_FSAvailableSpace,10,768.0,768.0,768.0\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity,15,15.416955,15.417,15.417\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPercent,15,0.471497,0.4715,0.4715\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-boot_FSUsedSpace,10,140.0,140.0,140.0\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSAvailableSpace,10,3730.0,3730.0,3730.0\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapacity,15,0.4268,0.4268,0.426812\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInodeUsedPercent,15,0.004196,0.0042,0.0042\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSUsedSpace,10,15.0,15.0,15.0\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-home_FSAvailableSpace,10,9207.0,9207.0,9207.0\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-home_FSCapacity,15,2.949693,2.949759,2.9498\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-home_FSInodeUsedPercent,15,0.211792,0.2118,0.2118\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-home_FSUsedSpace,10,279.0,279.0,279.0\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-tmp_FSAvailableSpace,10,4657.0,4657.0,4657.0\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-tmp_FSCapacity,15,0.98727,0.9873,0.9873\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-tmp_FSInodeUsedPercent,15,0.015869,0.0159,0.0159\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-tmp_FSUsedSpace,10,46.0,46.0,46.0\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailableSpace,10,34958.5,34992.0,35015.3\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-tomcat_FSCapacity,15,26.385434,26.449291,26.514927\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-tomcat_FSInodeUsedPercent,15,0.090495,0.090637,0.090844\nmetric,IG01,OSLinux-OSLinux_FILESYSTEM_-tomcat_FSUsedSpace,10,12548.15,12572.0,12604.5\nmetric,IG01,OSLinux-OSLinux_FILE_-home-zabbix_DirSizeMB,1,172.0,172.0,172.0\nmetric,IG01,OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_FileSizeMB,17,0.3138,0.3139,0.3151\nmetric,IG01,OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvgServ,7,0.85,0.85,0.85\nmetric,IG01,OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps,15,0.0,0.0,0.3\nmetric,IG01,OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPercentBusy,15,0.0,0.0,0.3\nmetric,IG01,OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps,15,0.0,0.0,0.06\nmetric,IG01,OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead,15,0.0,0.0,1.83999\nmetric,IG01,OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite,12,7.0,8.5,12.45\nmetric,IG01,OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps,11,0.0,0.0,0.0\n', 'file,cmdb_id,trace_name,count,p05,p50,p95\ntrace,IG01,trace.from_root.duration_mean,1,0.0365,0.0365,0.0365\ntrace,IG01,trace.from_root.duration_p95,1,0.04505,0.04505,0.04505\ntrace,IG01,trace.from_root.row_count,1,2.0,2.0,2.0\ntrace,IG01,trace.self.duration_mean,30,0.996396,6.109118,6.843902\ntrace,IG01,trace.self.duration_p95,30,6.27115,15.429225,16.062462\ntrace,IG01,trace.self.row_count,30,656.9,804.0,1449.4\ntrace,IG01,trace.to_Tomcat01.duration_mean,30,1.142603,5.919318,7.02134\ntrace,IG01,trace.to_Tomcat01.duration_p95,30,6.26286,15.29495,16.115175\ntrace,IG01,trace.to_Tomcat01.row_count,30,75.35,106.0,181.1\ntrace,IG01,trace.to_Tomcat02.duration_mean,30,1.134113,5.959507,7.226322\ntrace,IG01,trace.to_Tomcat02.duration_p95,30,6.821575,15.338175,16.030055\ntrace,IG01,trace.to_Tomcat02.row_count,30,77.35,101.0,172.35\ntrace,IG01,trace.to_Tomcat03.duration_mean,30,0.911254,6.189936,7.233863\ntrace,IG01,trace.to_Tomcat03.duration_p95,30,6.2302,15.2301,15.750342\ntrace,IG01,trace.to_Tomcat03.row_count,30,80.25,98.0,180.1\ntrace,IG01,trace.to_Tomcat04.duration_mean,30,0.860202,6.173278,7.359501\ntrace,IG01,trace.to_Tomcat04.duration_p95,30,5.264818,15.373525,16.289068\ntrace,IG01,trace.to_Tomcat04.row_count,30,52.55,99.5,181.75\ntrace,IG02,trace.from_root.duration_mean,1,5.029,5.029,5.029\ntrace,IG02,trace.from_root.duration_p95,1,9.5182,9.5182,9.5182\ntrace,IG02,trace.from_root.row_count,1,2.0,2.0,2.0\ntrace,IG02,trace.self.duration_mean,30,0.951502,6.072274,6.972224\ntrace,IG02,trace.self.duration_p95,30,6.212448,15.40045,16.089838\ntrace,IG02,trace.self.row_count,30,691.4,799.0,1516.8\ntrace,IG02,trace.to_Tomcat01.duration_mean,30,0.825223,5.818834,6.974194\ntrace,IG02,trace.to_Tomcat01.duration_p95,30,5.917505,13.575,16.143245\ntrace,IG02,trace.to_Tomcat01.row_count,30,81.9,102.5,186.65\ntrace,IG02,trace.to_Tomcat02.duration_mean,30,0.913554,5.917068,7.193475\ntrace,IG02,trace.to_Tomcat02.duration_p95,30,6.070598,15.3558,16.29974\ntrace,IG02,trace.to_Tomcat02.row_count,30,79.05,98.5,184.15\ntrace,IG02,trace.to_Tomcat03.duration_mean,30,1.119322,5.971621,7.178365\ntrace,IG02,trace.to_Tomcat03.duration_p95,30,5.78638,15.3735,16.204255\ntrace,IG02,trace.to_Tomcat03.row_count,30,81.35,101.0,185.65\ntrace,IG02,trace.to_Tomcat04.duration_mean,30,1.090701,6.135946,7.140169\ntrace,IG02,trace.to_Tomcat04.duration_p95,30,6.528175,15.343,16.329908\ntrace,IG02,trace.to_Tomcat04.row_count,30,53.05,100.0,176.4\ntrace,MG01,trace.from_Tomcat01.duration_mean,30,0.839469,8.006832,9.335021\ntrace,MG01,trace.from_Tomcat01.duration_p95,30,3.672795,15.628275,15.646882\ntrace,MG01,trace.from_Tomcat01.row_count,30,76.0,103.5,182.55\ntrace,MG01,trace.from_Tomcat02.duration_mean,30,1.059625,8.655819,10.696819\ntrace,MG01,trace.from_Tomcat02.duration_p95,30,8.63579,15.615825,15.643932\ntrace,MG01,trace.from_Tomcat02.row_count,30,78.8,98.5,180.1\ntrace,MG01,trace.from_Tomcat03.duration_mean,30,1.22933,8.425721,10.174319\ntrace,MG01,trace.from_Tomcat03.duration_p95,30,8.18865,15.599,15.63918\ntrace,MG01,trace.from_Tomcat03.row_count,30,80.15,102.0,188.55\ntrace,MG01,trace.from_Tomcat04.duration_mean,30,0.856473,8.401921,10.084163\ntrace,MG01,trace.from_Tomcat04.duration_p95,30,3.74983,15.612125,15.652582\ntrace,MG01,trace.from_Tomcat04.row_count,30,50.45,96.5,173.0\ntrace,MG01,trace.from_dockerA1.duration_mean,29,0.072556,7.641,15.32412\ntrace,MG01,trace.from_dockerA1.duration_p95,29,0.1266,15.443,15.60256\n', 'file,cmdb_id,log_name,count,p05,p50,p95\nlog,Mysql01,log.error_count,21,0.0,0.0,0.0\nlog,Mysql01,log.row_count,21,1.0,2.0,4.0\nlog,Mysql02,log.error_count,1,0.0,0.0,0.0\nlog,Mysql02,log.row_count,1,1.0,1.0,1.0\nlog,Tomcat01,log.error_count,31,0.0,0.0,0.0\nlog,Tomcat01,log.gc_pause_max,24,0.0215,0.07,0.09\nlog,Tomcat01,log.row_count,31,153.5,199.0,356.5\nlog,Tomcat02,log.error_count,31,0.0,0.0,0.0\nlog,Tomcat02,log.gc_pause_max,30,0.0345,0.07,0.171\nlog,Tomcat02,log.row_count,31,163.0,216.0,351.5\nlog,Tomcat03,log.error_count,31,0.0,0.0,0.0\nlog,Tomcat03,log.gc_pause_max,24,0.02,0.06,0.09\nlog,Tomcat03,log.row_count,31,168.5,201.0,368.0\nlog,Tomcat04,log.error_count,31,0.0,0.0,0.0\nlog,Tomcat04,log.gc_pause_max,22,0.05,0.07,0.09\nlog,Tomcat04,log.row_count,31,170.0,208.0,349.0\nlog,apache01,log.error_count,31,0.0,0.0,0.0\nlog,apache01,log.row_count,31,340.0,399.0,751.0\nlog,apache02,log.error_count,31,0.0,0.0,0.0\nlog,apache02,log.row_count,31,325.5,402.0,735.5\n', 'timestamp,cmdb_id,message\n2021-03-10T17:59:24+0800,Tomcat04,IG01 POST /UOCP/person/ServiceTest9.json HTTP/1.1 500 1194 - k6/0.29.0 (api_url 60011 60.011\n')```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse metric_df from previous state; define candidate components
candidates = [
    "apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"
]

# Filter metric rows to only candidate components (no time filtering)
metric_candidates = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# Group by (cmdb_id, kpi_name) and compute count and global percentiles (p05, p50, p95)
grp = metric_candidates.groupby(['cmdb_id', 'kpi_name'])['value']

def pct05(x):
    return float(np.nanpercentile(x.values, 5)) if x.notna().any() else np.nan

def pct50(x):
    return float(np.nanpercentile(x.values, 50)) if x.notna().any() else np.nan

def pct95(x):
    return float(np.nanpercentile(x.values, 95)) if x.notna().any() else np.nan

summary = grp.agg(
    count='count',
    p05=pct05,
    p50=pct50,
    p95=pct95
).reset_index()

# Round percentiles for compactness
summary[['p05','p50','p95']] = summary[['p05','p50','p95']].round(6)

# Sort by cmdb_id then kpi_name and limit to top 50 rows
summary_sorted = summary.sort_values(['cmdb_id','kpi_name']).head(50)

# Convert to CSV-like string
result_csv = summary_sorted.to_csv(index=False)

# Display the CSV-like result string (compact)
result_csv

```
Out[2]:
```
Summary of the computed global percentiles (full series) restricted to the candidate root-cause components:

- Returned rows: The CSV output shown contains the top 50 rows sorted by cmdb_id and kpi_name — these rows are all for component IG01 (other candidate components’ metrics exist but were not shown because of the 50-row limit).

Key observations from IG01 metrics (count, p05, p50 (median), p95):

- JVM memory:
  - JVM HeapMemoryMax (count=31): constant 2,040,135,680.
  - HeapMemoryUsage (count=20): p05=6.283475, median=22.9956, p95=38.825065.
  - HeapMemoryUsed (count=31): p05=106,814,508, median=478,570,128, p95=842,213,152.
  - NoHeapMemoryUsed (count=31): constant 57,925,184.

- JVM & CPU:
  - JVM CPULoad (count=20): median ~0.25585 (p95 ~0.41252).
  - CPU overall util (CPUCpuUtil, count=21): median ~25.8502 (p05 ~25.6953, p95 ~26.1153).
  - CPU user time and sys time show small values consistent with low OS-level CPU sys% but overall CPU util ~25%.

- Filesystem (many per-mount metrics):
  - tomcat filesystem used space (count=10): median ~12,572 MB (p05 ~12,548.15, p95 ~12,604.5).
  - Several FS available/used values are stable (e.g., FSAvailableSpace, FSUsedSpace) with identical percentiles (indicating near-constant readings).

- Disk I/O (sda):
  - DSKRead (count=15): p95 ~1.83999, medians often 0.0 (low read activity).
  - DSKReadWrite and DSKAvgServ show small values (limited activity).

- Misc:
  - Some file-size metrics and inode percentages are effectively constant across samples.
  - Group sample sizes vary (counts seen from 1 up to 31 depending on KPI).

Conclusion: The provided CSV rows show IG01’s global percentiles across many JVM, CPU, filesystem, and disk KPIs. Key values to note are moderate heap usage (median ~478 MB used; usage ~23%), CPU overall util around ~25.8% median, and tomcat filesystem used ~12.6 GB. If you want the same percentile table for other candidate components (apache*, Tomcat*, MG*, Mysql*, Redis*), I can return the next rows or produce separate tables limited per component.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id,kpi_name,count,p05,p50,p95
IG01,JVM-Memory_7778_JVM_Memory_HeapMemoryMax,31,2040135680.0,2040135680.0,2040135680.0
IG01,JVM-Memory_7778_JVM_Memory_HeapMemoryUsage,20,6.283475,22.9956,38.825065
IG01,JVM-Memory_7778_JVM_Memory_HeapMemoryUsed,31,106814508.0,478570128.0,842213152.0
IG01,JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed,31,57925184.0,57925184.0,57925184.0
IG01,JVM-Operating System_7778_JVM_JVM_CPULoad,20,0.220125,0.25585,0.41252
IG01,JVM-Runtime_7778_JVM_JVM_Uptime,21,447704.0,448544.0,449325.0
IG01,JVM-Threads_7778_JVM_ThreadCount_Threads,31,67.0,71.0,75.0
IG01,OSLinux-CPU_CPU-0_SingleCpuUtil,11,0.78185,1.0476,15.6558
IG01,OSLinux-CPU_CPU-0_SingleCpuidle,15,89.148951,98.9192,99.204764
IG01,OSLinux-CPU_CPU-2_SingleCpuUtil,11,0.84835,1.0484,2.32955
IG01,OSLinux-CPU_CPU-2_SingleCpuidle,15,97.22867,98.951573,99.144883
IG01,OSLinux-CPU_CPU_CPUCpuUtil,21,25.6953,25.8502,26.1153
IG01,OSLinux-CPU_CPU_CPULoad,31,1.0,1.03,1.135
IG01,OSLinux-CPU_CPU_CPUSysTime,31,0.291,0.353416,0.430253
IG01,OSLinux-CPU_CPU_CPUUserTime,31,25.1653,25.21,25.319608
IG01,OSLinux-CPU_CPU_CPUWio,31,0.0166,0.06652,0.4802
IG01,OSLinux-CPU_CPU_CPUidleutil,31,73.678528,74.18,74.308342
IG01,OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace,10,9951.0,9951.0,9951.0
IG01,OSLinux-OSLinux_FILESYSTEM_-_FSCapacity,15,30.2645,30.264562,30.2646
IG01,OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent,15,16.366374,16.3664,16.3664
IG01,OSLinux-OSLinux_FILESYSTEM_-_FSUsedSpace,10,4319.0,4319.0,4319.0
IG01,OSLinux-OSLinux_FILESYSTEM_-boot_FSAvailableSpace,10,768.0,768.0,768.0
IG01,OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity,15,15.416955,15.417,15.417
IG01,OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPercent,15,0.471497,0.4715,0.4715
IG01,OSLinux-OSLinux_FILESYSTEM_-boot_FSUsedSpace,10,140.0,140.0,140.0
IG01,OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSAvailableSpace,10,3730.0,3730.0,3730.0
IG01,OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapacity,15,0.4268,0.4268,0.426812
IG01,OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInodeUsedPercent,15,0.004196,0.0042,0.0042
IG01,OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSUsedSpace,10,15.0,15.0,15.0
IG01,OSLinux-OSLinux_FILESYSTEM_-home_FSAvailableSpace,10,9207.0,9207.0,9207.0
IG01,OSLinux-OSLinux_FILESYSTEM_-home_FSCapacity,15,2.949693,2.949759,2.9498
IG01,OSLinux-OSLinux_FILESYSTEM_-home_FSInodeUsedPercent,15,0.211792,0.2118,0.2118
IG01,OSLinux-OSLinux_FILESYSTEM_-home_FSUsedSpace,10,279.0,279.0,279.0
IG01,OSLinux-OSLinux_FILESYSTEM_-tmp_FSAvailableSpace,10,4657.0,4657.0,4657.0
IG01,OSLinux-OSLinux_FILESYSTEM_-tmp_FSCapacity,15,0.98727,0.9873,0.9873
IG01,OSLinux-OSLinux_FILESYSTEM_-tmp_FSInodeUsedPercent,15,0.015869,0.0159,0.0159
IG01,OSLinux-OSLinux_FILESYSTEM_-tmp_FSUsedSpace,10,46.0,46.0,46.0
IG01,OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailableSpace,10,34958.5,34992.0,35015.3
IG01,OSLinux-OSLinux_FILESYSTEM_-tomcat_FSCapacity,15,26.385434,26.449291,26.514927
IG01,OSLinux-OSLinux_FILESYSTEM_-tomcat_FSInodeUsedPercent,15,0.090495,0.090637,0.090844
IG01,OSLinux-OSLinux_FILESYSTEM_-tomcat_FSUsedSpace,10,12548.15,12572.0,12604.5
IG01,OSLinux-OSLinux_FILE_-home-zabbix_DirSizeMB,1,172.0,172.0,172.0
IG01,OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_FileSizeMB,17,0.3138,0.3139,0.3151
IG01,OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvgServ,7,0.85,0.85,0.85
IG01,OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps,15,0.0,0.0,0.3
IG01,OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPercentBusy,15,0.0,0.0,0.3
IG01,OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps,15,0.0,0.0,0.06
IG01,OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead,15,0.0,0.0,1.83999
IG01,OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite,12,7.0,8.5,12.45
IG01,OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps,11,0.0,0.0,0.0```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df from previous state; ensure timezone-aware ts exists
tz = pytz.timezone('Asia/Shanghai')
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # Ensure ts is timezone-aware in Asia/Shanghai
    if metric_df['ts'].dt.tz is None:
        metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Candidate components
candidates = [
    "apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"
]

# 1) Compute global p95 per (cmdb_id, kpi_name) using full series (no time filter)
metric_candidates = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

def compute_percentiles(group):
    values = group.values
    return pd.Series({
        'count': int(np.count_nonzero(~np.isnan(values))),
        'global_p95': float(np.nanpercentile(values, 95)) if np.nanmin(values) is not np.nan else np.nan
    })

p95_df = metric_candidates.groupby(['cmdb_id','kpi_name'])['value'].apply(compute_percentiles).unstack().reset_index()
# The above produced MultiIndex columns; fix it if needed
if isinstance(p95_df.columns, pd.MultiIndex):
    p95_df.columns = ['cmdb_id','kpi_name','count','global_p95']
else:
    # ensure columns are correct
    p95_df = p95_df.rename(columns={0:'count',1:'global_p95'}) if 'global_p95' not in p95_df.columns else p95_df

p95_df = p95_df[['cmdb_id','kpi_name','count','global_p95']]
p95_df['global_p95'] = p95_df['global_p95'].astype(float).round(6)

# 2) Filter to incident window in UTC+8: 2021-03-10 17:30:00 to 2021-03-10 18:00:00 (inclusive)
start = pd.Timestamp('2021-03-10 17:30:00').tz_localize(tz)
end = pd.Timestamp('2021-03-10 18:00:00').tz_localize(tz)

metric_window = metric_candidates[(metric_candidates['ts'] >= start) & (metric_candidates['ts'] <= end)].copy()

# 2b) For each group, find anomaly points where value >= global_p95
# Merge thresholds into windowed data
merged = metric_window.merge(p95_df[['cmdb_id','kpi_name','global_p95']], on=['cmdb_id','kpi_name'], how='left')

# Mark anomalies
merged['is_anomaly'] = merged['value'] >= merged['global_p95']

# Aggregate per group to get anomaly_count, earliest anomaly timestamp, max anomaly value
agg = merged[merged['is_anomaly']].groupby(['cmdb_id','kpi_name']).agg(
    anomaly_count = ('value','count'),
    earliest_ts = ('ts','min'),
    max_anomaly_value = ('value','max')
).reset_index()

# Ensure groups with zero anomalies are also present (with zeros/NaNs)
all_groups = p95_df[['cmdb_id','kpi_name','global_p95']].copy()
summary = all_groups.merge(agg, on=['cmdb_id','kpi_name'], how='left')
summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
summary['earliest_ts'] = pd.to_datetime(summary['earliest_ts']).dt.tz_convert(tz)
# Format earliest timestamp as ISO+0800 or empty if no anomaly
summary['earliest_anomaly_timestamp'] = summary['earliest_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
summary.loc[summary['anomaly_count']==0,'earliest_anomaly_timestamp'] = ''

summary['max_anomaly_value'] = summary['max_anomaly_value'].round(6)
summary.loc[summary['anomaly_count']==0,'max_anomaly_value'] = np.nan

# Select required columns and sort by anomaly_count desc then cmdb_id
out = summary[['cmdb_id','kpi_name','global_p95','anomaly_count','earliest_anomaly_timestamp','max_anomaly_value']].copy()
out_sorted = out.sort_values(['anomaly_count','cmdb_id'], ascending=[False, True]).head(20)

# 3) For the single most anomalous row, return up to 10 anomaly points (timestamp,value) sorted by timestamp
most_anomalous = out_sorted.iloc[0] if out_sorted.shape[0] > 0 else None

anomaly_points_df = pd.DataFrame(columns=['timestamp','value'])
if most_anomalous is not None and most_anomalous['anomaly_count'] > 0:
    cm = most_anomalous['cmdb_id']
    kp = most_anomalous['kpi_name']
    gp95 = most_anomalous['global_p95']
    group_points = merged[(merged['cmdb_id']==cm) & (merged['kpi_name']==kp) & (merged['is_anomaly'])].copy()
    # Sort by ts and limit to 10
    group_points = group_points.sort_values('ts').head(10)
    anomaly_points_df = group_points[['ts','value']].copy()
    anomaly_points_df['timestamp'] = anomaly_points_df['ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    anomaly_points_df = anomaly_points_df[['timestamp','value']].reset_index(drop=True)

# Convert summaries to CSV-like strings for compact display
summary_csv = out_sorted.to_csv(index=False)
anomalies_csv = anomaly_points_df.to_csv(index=False)

# Display the two compact CSV-like outputs
summary_csv, anomalies_csv

```
Out[3]:
```
Summary of results (metric anomalies within 2021-03-10 17:30–18:00 UTC+8, using global p95 thresholds):

Key findings
- Many groups triggered anomalies for the entire incident window (anomaly_count = 31), meaning every sample in the 30-minute window was >= that group's global p95.
- Top groups (highest anomaly_count) include:
  - IG01, JVM-Memory_7778_JVM_Memory_HeapMemoryMax — global_p95 = 2040135680.0, anomaly_count = 31, earliest = 2021-03-10T17:30:00+0800, max_anomaly_value = 2040135680.0
  - IG01, JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed — global_p95 = 57925184.0, anomaly_count = 31
  - IG01, several OSLinux network/process metrics (e.g., TCP-CLOSE-WAIT, TCP-FIN-WAIT, PROCESS_PROCNoZombies, zabbix process pct) — all anomaly_count = 31
  - IG02, JVM-Memory_7778_JVM_Memory_HeapMemoryMax — global_p95 = 2040135680.0, anomaly_count = 31
  - MG01 and MG02 JVM heap/no-heap max metrics also show anomaly_count = 31
  - Mysql01, Mysql02 network/process metrics also appear with anomaly_count = 31
- The summary is limited to the top 20 rows sorted by anomaly_count and cmdb_id; many entries show the same pattern (full-window anomalies).

Most anomalous group (highest anomaly_count)
- Group: IG01, JVM-Memory_7778_JVM_Memory_HeapMemoryMax
- global_p95: 2040135680.0
- anomaly_count: 31 (every sample in window)
- earliest anomaly: 2021-03-10T17:30:00+0800
- max anomaly value: 2040135680.0

Example anomaly points for that group (first up to 10 timestamps)
- 2021-03-10T17:30:00+0800, 2040135680.0
- 2021-03-10T17:31:00+0800, 2040135680.0
- 2021-03-10T17:32:00+0800, 2040135680.0
- 2021-03-10T17:33:00+0800, 2040135680.0
- 2021-03-10T17:34:00+0800, 2040135680.0
- 2021-03-10T17:35:00+0800, 2040135680.0
- 2021-03-10T17:36:00+0800, 2040135680.0
- 2021-03-10T17:37:00+0800, 2040135680.0
- 2021-03-10T17:38:00+0800, 2040135680.0
- 2021-03-10T17:39:00+0800, 2040135680.0

Interpretation & suggested next steps
- Many of these "anomalies" are due to metrics being equal to a constant high value (e.g., HeapMemoryMax) or network/process counters that are consistently at/above their global p95; when global p95 equals a constant, every point at that constant will be counted as an anomaly.
- Investigate IG01/IG02 and MG nodes for JVM memory configuration and why HeapMemoryMax values are at that constant. Check whether "HeapMemoryMax" being equal to global p95 is meaningful (it may simply reflect the configured maximum heap size rather than a runtime issue).
- Correlate with application logs and traces (there is an earlier error log showing a Tomcat04 HTTP 500) to see if these metric anomalies align with service errors.
- If the goal is to find unusual runtime spikes rather than constant configuration values, consider excluding constant-capacity metrics (like HeapMemoryMax) or using a different anomaly rule (e.g., value >= p95 and > p50 by some margin).

If you want, I can:
- Return the full top-20 anomaly table as CSV text, or
- Recompute using a stricter anomaly rule (e.g., value > global_p95 and > median + delta) to filter out constant-value thresholds.

The original code execution output of IPython Kernel is also provided below for reference:

('cmdb_id,kpi_name,global_p95,anomaly_count,earliest_anomaly_timestamp,max_anomaly_value\nIG01,JVM-Memory_7778_JVM_Memory_HeapMemoryMax,2040135680.0,31,2021-03-10T17:30:00+0800,2040135680.0\nIG01,JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed,57925184.0,31,2021-03-10T17:30:00+0800,57925184.0\nIG01,OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT,0.0,31,2021-03-10T17:30:00+0800,0.0\nIG01,OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT,0.0,31,2021-03-10T17:30:00+0800,0.0\nIG01,OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies,1.0,31,2021-03-10T17:30:00+0800,1.0\nIG01,OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc,0.3,31,2021-03-10T17:30:00+0800,0.3\nIG01,OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount,3.0,31,2021-03-10T17:30:00+0800,3.0\nIG02,JVM-Memory_7778_JVM_Memory_HeapMemoryMax,2040135680.0,31,2021-03-10T17:30:00+0800,2040135680.0\nIG02,OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT,0.0,31,2021-03-10T17:30:00+0800,0.0\nIG02,OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies,1.0,31,2021-03-10T17:30:00+0800,1.0\nIG02,OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount,3.0,31,2021-03-10T17:30:00+0800,3.0\nMG01,JVM-Memory_7779_JVM_Memory_HeapMemoryMax,2040135680.0,31,2021-03-10T17:30:00+0800,2040135680.0\nMG01,JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed,506945232.0,31,2021-03-10T17:30:00+0800,506945232.0\nMG02,JVM-Memory_7779_JVM_Memory_HeapMemoryMax,2040135680.0,31,2021-03-10T17:30:00+0800,2040135680.0\nMG02,JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed,947152056.0,31,2021-03-10T17:30:00+0800,947152056.0\nMysql01,OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT,0.0,31,2021-03-10T17:30:00+0800,0.0\nMysql01,OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT,0.0,31,2021-03-10T17:30:00+0800,1.0\nMysql01,OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies,0.0,31,2021-03-10T17:30:00+0800,0.0\nMysql01,OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount,7.0,31,2021-03-10T17:30:00+0800,7.0\nMysql02,OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT,0.0,31,2021-03-10T17:30:00+0800,0.0\n', 'timestamp,value\n2021-03-10T17:30:00+0800,2040135680.0\n2021-03-10T17:31:00+0800,2040135680.0\n2021-03-10T17:32:00+0800,2040135680.0\n2021-03-10T17:33:00+0800,2040135680.0\n2021-03-10T17:34:00+0800,2040135680.0\n2021-03-10T17:35:00+0800,2040135680.0\n2021-03-10T17:36:00+0800,2040135680.0\n2021-03-10T17:37:00+0800,2040135680.0\n2021-03-10T17:38:00+0800,2040135680.0\n2021-03-10T17:39:00+0800,2040135680.0\n')```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_df and ts exist (reuse from kernel state)
if 'trace_df' not in globals():
    trace_df = pd.read_csv('trace.csv')
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware in Asia/Shanghai
    if trace_df['ts'].dt.tz is None:
        trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global p95 per (cmdb_id, trace_name) using full series
p95_df = trace_df.groupby(['cmdb_id','trace_name'])['value'].agg(
    count='count',
    global_p95=lambda x: float(np.nanpercentile(x.values,95))
).reset_index()
p95_df['global_p95'] = p95_df['global_p95'].round(6)

# 2) Filter to incident window (UTC+8) 2021-03-10 17:30:00 — 2021-03-10 18:00:00
start = pd.Timestamp('2021-03-10 17:30:00').tz_localize(tz)
end = pd.Timestamp('2021-03-10 18:00:00').tz_localize(tz)
trace_window = trace_df[(trace_df['ts'] >= start) & (trace_df['ts'] <= end)].copy()

# Merge thresholds and mark anomalies (value >= global_p95)
merged = trace_window.merge(p95_df[['cmdb_id','trace_name','global_p95']], on=['cmdb_id','trace_name'], how='left')
merged['is_anomaly'] = merged['value'] >= merged['global_p95']

# 3) Aggregate anomalies per group
anoms = merged[merged['is_anomaly']].groupby(['cmdb_id','trace_name']).agg(
    anomaly_count = ('value','count'),
    earliest_ts = ('ts','min'),
    max_anomaly_value = ('value','max')
).reset_index()

# Attach global_p95 for output
summary = anoms.merge(p95_df[['cmdb_id','trace_name','global_p95']], on=['cmdb_id','trace_name'], how='left')

# Format earliest timestamp and round values
summary['earliest_anomaly_timestamp'] = summary['earliest_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
summary['max_anomaly_value'] = summary['max_anomaly_value'].round(6)
summary['global_p95'] = summary['global_p95'].round(6)

# Select and sort top 20 by anomaly_count desc
summary_out = summary[['cmdb_id','trace_name','global_p95','anomaly_count','earliest_anomaly_timestamp','max_anomaly_value']].sort_values(
    ['anomaly_count','cmdb_id'], ascending=[False, True]
).head(20).reset_index(drop=True)

# 4) For the single top anomalous group, return up to 10 anomaly points (timestamp,value)
anomaly_points_df = pd.DataFrame(columns=['timestamp','value'])
if summary_out.shape[0] > 0:
    top = summary_out.iloc[0]
    cm = top['cmdb_id']
    tn = top['trace_name']
    group_points = merged[(merged['cmdb_id']==cm) & (merged['trace_name']==tn) & (merged['is_anomaly'])].copy()
    if not group_points.empty:
        group_points = group_points.sort_values('ts').head(10)
        anomaly_points_df = group_points[['ts','value']].copy()
        anomaly_points_df['timestamp'] = anomaly_points_df['ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
        anomaly_points_df = anomaly_points_df[['timestamp','value']].reset_index(drop=True)

# Convert to compact CSV-like strings
summary_csv = summary_out.to_csv(index=False)
anomalies_csv = anomaly_points_df.to_csv(index=False)

# Display compact results
summary_csv, anomalies_csv

```
Out[4]:
```
Summary of trace anomalies during the incident window (2021-03-10 17:30–18:00 UTC+8):

- Top anomalous group: Tomcat04, trace.self.duration_p95
  - Global p95 = 10.011
  - anomaly_count = 15 (most occurrences in the window)
  - earliest anomaly = 2021-03-10T17:33:00+0800
  - max anomaly value = 10.011
  - Example anomaly points (up to 10): 17:33, 17:35, 17:37, 17:38, 17:39, 17:40, 17:42, 17:47, 17:48, 17:49 (all with value 10.011)

- Other notable groups (top by anomaly_count):
  - Tomcat02, trace.self.duration_p95 — global_p95 ≈ 10.011, anomaly_count = 12, earliest 17:33
  - Tomcat03, trace.self.duration_p95 — global_p95 ≈ 10.011, anomaly_count = 11, earliest 17:33
  - Tomcat01, trace.self.duration_p95 — global_p95 ≈ 10.011, anomaly_count = 8, earliest 17:33
  - MG01 and Tomcat04 row_count links (trace.from_Tomcat04.row_count and trace.to_MG01.row_count) — anomaly_count = 3, earliest 17:30, max ~204.0
  - IG01 shows several smaller-rate anomalies (duration_mean, duration_p95, row_count, to_Tomcat* metrics) with anomaly_count = 2 each.

Interpretation and next steps:
- The dominant signal is elevated p95 trace durations on Tomcat nodes (Tomcat04 highest), with repeated hits at or above the historical global p95 during the incident window — this suggests tail latency spikes in Tomcat services.
- MG01 row_count and Tomcat04→MG01 row_count anomalies indicate increased request/row counts that may correlate with the latency spikes.
- Correlate these trace anomalies with application logs and metric anomalies already observed (there is a Tomcat04 HTTP 500 in error_logs) to determine cause (e.g., request load, downstream slowness, GC/CPU pressure).
- Recommended actions: inspect Tomcat04 logs and GC/CPU/memory metrics around the listed timestamps, check MG01 downstream performance, and review recent deploys/config changes for Tomcat services.

The original code execution output of IPython Kernel is also provided below for reference:

('cmdb_id,trace_name,global_p95,anomaly_count,earliest_anomaly_timestamp,max_anomaly_value\nTomcat04,trace.self.duration_p95,10.011,15,2021-03-10T17:33:00+0800,10.011\nTomcat02,trace.self.duration_p95,10.011,12,2021-03-10T17:33:00+0800,10.012\nTomcat03,trace.self.duration_p95,10.011,11,2021-03-10T17:33:00+0800,10.011\nTomcat01,trace.self.duration_p95,10.011,8,2021-03-10T17:33:00+0800,10.011\nMG01,trace.from_Tomcat04.row_count,173.0,3,2021-03-10T17:30:00+0800,204.0\nTomcat04,trace.to_MG01.row_count,173.0,3,2021-03-10T17:30:00+0800,204.0\nIG01,trace.self.duration_mean,6.843902,2,2021-03-10T17:33:00+0800,6.919549\nIG01,trace.self.duration_p95,16.062462,2,2021-03-10T17:39:00+0800,16.2202\nIG01,trace.self.row_count,1449.4,2,2021-03-10T17:30:00+0800,1574.0\nIG01,trace.to_Tomcat01.duration_mean,7.02134,2,2021-03-10T17:50:00+0800,7.201913\nIG01,trace.to_Tomcat01.duration_p95,16.115175,2,2021-03-10T17:50:00+0800,16.3239\nIG01,trace.to_Tomcat01.row_count,181.1,2,2021-03-10T17:30:00+0800,197.0\nIG01,trace.to_Tomcat02.duration_mean,7.226322,2,2021-03-10T17:41:00+0800,7.584057\nIG01,trace.to_Tomcat02.duration_p95,16.030055,2,2021-03-10T17:39:00+0800,16.27565\nIG01,trace.to_Tomcat02.row_count,172.35,2,2021-03-10T17:30:00+0800,205.0\nIG01,trace.to_Tomcat03.duration_mean,7.233863,2,2021-03-10T17:34:00+0800,7.265699\nIG01,trace.to_Tomcat03.duration_p95,15.750342,2,2021-03-10T17:39:00+0800,16.22335\nIG01,trace.to_Tomcat03.row_count,180.1,2,2021-03-10T17:31:00+0800,206.0\nIG01,trace.to_Tomcat04.duration_mean,7.359501,2,2021-03-10T17:33:00+0800,7.564262\nIG01,trace.to_Tomcat04.duration_p95,16.289068,2,2021-03-10T17:41:00+0800,16.49875\n', 'timestamp,value\n2021-03-10T17:33:00+0800,10.011\n2021-03-10T17:35:00+0800,10.011\n2021-03-10T17:37:00+0800,10.011\n2021-03-10T17:38:00+0800,10.011\n2021-03-10T17:39:00+0800,10.011\n2021-03-10T17:40:00+0800,10.011\n2021-03-10T17:42:00+0800,10.011\n2021-03-10T17:47:00+0800,10.011\n2021-03-10T17:48:00+0800,10.011\n2021-03-10T17:49:00+0800,10.011\n')```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing DataFrames: log_df, error_logs_df (created in previous steps)
tz = pytz.timezone('Asia/Shanghai')

# Ensure timezone-aware timestamps exist
if 'ts' not in log_df.columns:
    log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if log_df['ts'].dt.tz is None:
        log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

if 'ts' not in error_logs_df.columns:
    error_logs_df['ts'] = pd.to_datetime(error_logs_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if error_logs_df['ts'].dt.tz is None:
        error_logs_df['ts'] = pd.to_datetime(error_logs_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window (inclusive) in UTC+8
start = pd.Timestamp('2021-03-10 17:30:00').tz_localize(tz)
end = pd.Timestamp('2021-03-10 18:00:00').tz_localize(tz)

# 1) Compute global p05 and p95 per (cmdb_id, log_name) using full series (no time filtering)
agg_thresholds = log_df.groupby(['cmdb_id','log_name'])['value'].agg(
    count='count',
    global_p05=lambda x: float(np.nanpercentile(x.values, 5)) if x.notna().any() else np.nan,
    global_p95=lambda x: float(np.nanpercentile(x.values, 95)) if x.notna().any() else np.nan
).reset_index()

agg_thresholds['global_p05'] = agg_thresholds['global_p05'].round(6)
agg_thresholds['global_p95'] = agg_thresholds['global_p95'].round(6)

# 2) Filter log.csv to the incident window and detect anomalies (value >= p95 OR value <= p05)
log_window = log_df[(log_df['ts'] >= start) & (log_df['ts'] <= end)].copy()

# Merge thresholds into windowed logs
merged = log_window.merge(agg_thresholds[['cmdb_id','log_name','global_p05','global_p95']], on=['cmdb_id','log_name'], how='left')

# Mark anomalies
merged['is_anomaly'] = (merged['value'] >= merged['global_p95']) | (merged['value'] <= merged['global_p05'])

# Aggregate anomalies per group
anoms = merged[merged['is_anomaly']].groupby(['cmdb_id','log_name']).agg(
    anomaly_count = ('value','count'),
    earliest_ts = ('ts','min'),
    max_anomaly_value = ('value','max')
).reset_index()

# Merge with thresholds to include global p05/p95, and ensure groups with zero anomalies appear
summary = agg_thresholds.merge(anoms, on=['cmdb_id','log_name'], how='left')
summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
summary['earliest_anomaly_timestamp'] = pd.to_datetime(summary['earliest_ts']).dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')
summary.loc[summary['anomaly_count']==0,'earliest_anomaly_timestamp'] = ''
summary['max_anomaly_value'] = summary['max_anomaly_value'].round(6)
summary.loc[summary['anomaly_count']==0,'max_anomaly_value'] = np.nan

# Select required columns and sort by anomaly_count desc, then cmdb_id, limit to top 20
summary_out = summary[['cmdb_id','log_name','global_p05','global_p95','anomaly_count','earliest_anomaly_timestamp','max_anomaly_value']].sort_values(
    ['anomaly_count','cmdb_id'], ascending=[False, True]
).head(20).reset_index(drop=True)

# 2) Up to 20 raw log rows for Tomcat04 within the window with columns: timestamp,log_name,value (sorted by timestamp)
tomcat04_logs = log_window[log_window['cmdb_id']=='Tomcat04'].copy().sort_values('ts').head(20)
if not tomcat04_logs.empty:
    tomcat04_logs_out = tomcat04_logs[['ts','log_name','value']].copy()
    tomcat04_logs_out['timestamp'] = tomcat04_logs_out['ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    tomcat04_logs_out = tomcat04_logs_out[['timestamp','log_name','value']].reset_index(drop=True)
else:
    tomcat04_logs_out = tomcat04_logs[['ts','log_name','value']].copy()  # empty

# 3) error_logs rows within the incident window sorted by timestamp, up to 50 rows
error_window = error_logs_df[(error_logs_df['ts'] >= start) & (error_logs_df['ts'] <= end)].copy().sort_values('ts').head(50)
if not error_window.empty:
    error_out = error_window[['ts','cmdb_id','message']].copy()
    error_out['timestamp'] = error_out['ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    error_out = error_out[['timestamp','cmdb_id','message']].reset_index(drop=True)
else:
    error_out = error_window[['ts','cmdb_id','message']].copy()  # empty

# Convert outputs to compact CSV-like strings
summary_csv = summary_out.to_csv(index=False)
tomcat04_logs_csv = tomcat04_logs_out.to_csv(index=False)
error_logs_csv = error_out.to_csv(index=False)

# Display compact CSV-like results
summary_csv, tomcat04_logs_csv, error_logs_csv

```
Out[5]:
```
Summary of log and error-log anomalies during the incident window (2021-03-10 17:30–18:00 UTC+8):

What was found (top signals)
- Many log.error_count series have global p05 = p95 = 0, so the rule (value >= p95 OR <= p05) marks almost every sample as an "anomaly" because the thresholds are 0. This produces large anomaly_count values (31) for Tomcat01/02/03/04, apache01/02, Mysql01, etc. This is an artifact of constant-zero thresholds, not necessarily meaningful errors. Note: Tomcat04 had a non-zero max error_count = 1.0 in the window (but still flagged for the 0 threshold logic).
- Meaningful log anomalies to inspect (spikes above typical):
  - GC pause spikes:
    - Tomcat01: log.gc_pause_max — anomaly_count 6, earliest 2021-03-10T17:33:00+0800, max 0.12 s
    - Tomcat03: log.gc_pause_max — anomaly_count 6, earliest 17:35:00, max 0.10 s
    - Tomcat04: log.gc_pause_max — anomaly_count 6, earliest 17:30:00, max 0.10 s
    - Tomcat02: log.gc_pause_max — anomaly_count 4, earliest 17:34:00, max 1.03 s (this large spike is notable)
  - Request/row count spikes:
    - Tomcat04: log.row_count — anomaly_count 4, earliest 17:30:00, max 424.0 (spike at start of window)
    - Tomcat02: log.row_count — anomaly_count 4, max 415.0 at 17:30
    - Tomcat01: log.row_count — anomaly_count 4, max 412.0 at 17:30
    - apache01, apache02: row_count spikes to ~789 and ~791 respectively at 17:30
    - Mysql01: log.row_count — anomaly_count 12, earliest 17:30, max 5.0 (smaller counts but repeated)
- Raw Tomcat04 logs within window (first rows shown) include row_count values like 424 at 17:30:00, gc_pause_max spikes at 17:32 (0.1) and 17:34 (0.08–0.09), and repeated error_count=0 at many minutes.
- error_logs.csv within the window: one entry found at 2021-03-10T17:59:24+0800 for Tomcat04 — an HTTP 500 for IG01 POST /UOCP/person/ServiceTest9.json (message truncated).

Interpretation and recommended next steps
- Many "anomalies" are due to threshold artifact (p95/p05 equals 0). Focus on the substantive anomalies: GC pause spikes (especially Tomcat02 with a 1.03s pause) and the large row_count bursts at ~17:30 across Tomcat and Apache nodes.
- Tomcat04 is a focal point: it shows row_count spike at 17:30, GC pause spikes, and has an HTTP 500 in error_logs at 17:59 — correlate these with the trace and metric anomalies already observed (trace p95 spikes on Tomcat nodes and IG duration anomalies).
- Actionable checks:
  - Inspect Tomcat04 logs (application and GC) around 17:30–17:40 and around 17:59.
  - Review request load patterns at ~17:30 (front-end apache counts and Tomcat row_count spikes) and whether backend MG/Mysql/Redis were overloaded.
  - Investigate Tomcat02’s large gc_pause_max = 1.03s at 17:34 — this could cause increased latency.
  - Use a refined anomaly rule (e.g., value > global_p95 AND value > median + delta) or exclude constant configuration metrics (p95=0) to reduce false positives.

If you want, I can:
- Provide the full top-20 anomaly CSV table as text, or
- Re-run anomaly detection using a stricter rule to filter out the constant-threshold artifacts.

The original code execution output of IPython Kernel is also provided below for reference:

('cmdb_id,log_name,global_p05,global_p95,anomaly_count,earliest_anomaly_timestamp,max_anomaly_value\nTomcat01,log.error_count,0.0,0.0,31,2021-03-10T17:30:00+0800,0.0\nTomcat02,log.error_count,0.0,0.0,31,2021-03-10T17:30:00+0800,0.0\nTomcat03,log.error_count,0.0,0.0,31,2021-03-10T17:30:00+0800,0.0\nTomcat04,log.error_count,0.0,0.0,31,2021-03-10T17:30:00+0800,1.0\napache01,log.error_count,0.0,0.0,31,2021-03-10T17:30:00+0800,0.0\napache02,log.error_count,0.0,0.0,31,2021-03-10T17:30:00+0800,0.0\nMysql01,log.error_count,0.0,0.0,21,2021-03-10T17:30:00+0800,0.0\nMysql01,log.row_count,1.0,4.0,12,2021-03-10T17:30:00+0800,5.0\nTomcat01,log.gc_pause_max,0.0215,0.09,6,2021-03-10T17:33:00+0800,0.12\nTomcat03,log.gc_pause_max,0.02,0.09,6,2021-03-10T17:35:00+0800,0.1\nTomcat04,log.gc_pause_max,0.05,0.09,6,2021-03-10T17:30:00+0800,0.1\nTomcat01,log.row_count,153.5,356.5,4,2021-03-10T17:30:00+0800,412.0\nTomcat02,log.gc_pause_max,0.0345,0.171,4,2021-03-10T17:34:00+0800,1.03\nTomcat02,log.row_count,163.0,351.5,4,2021-03-10T17:30:00+0800,415.0\nTomcat03,log.row_count,168.5,368.0,4,2021-03-10T17:30:00+0800,396.0\nTomcat04,log.row_count,170.0,349.0,4,2021-03-10T17:30:00+0800,424.0\napache01,log.row_count,340.0,751.0,4,2021-03-10T17:30:00+0800,789.0\napache02,log.row_count,325.5,735.5,4,2021-03-10T17:30:00+0800,791.0\nMysql02,log.error_count,0.0,0.0,1,2021-03-10T17:55:00+0800,0.0\nMysql02,log.row_count,1.0,1.0,1,2021-03-10T17:55:00+0800,1.0\n', 'timestamp,log_name,value\n2021-03-10T17:30:00+0800,log.error_count,0.0\n2021-03-10T17:30:00+0800,log.gc_pause_max,0.05\n2021-03-10T17:30:00+0800,log.row_count,424.0\n2021-03-10T17:31:00+0800,log.error_count,0.0\n2021-03-10T17:31:00+0800,log.row_count,358.0\n2021-03-10T17:32:00+0800,log.error_count,0.0\n2021-03-10T17:32:00+0800,log.gc_pause_max,0.1\n2021-03-10T17:32:00+0800,log.row_count,340.0\n2021-03-10T17:33:00+0800,log.row_count,197.0\n2021-03-10T17:33:00+0800,log.error_count,0.0\n2021-03-10T17:34:00+0800,log.error_count,0.0\n2021-03-10T17:34:00+0800,log.gc_pause_max,0.08\n2021-03-10T17:34:00+0800,log.row_count,192.0\n2021-03-10T17:35:00+0800,log.error_count,0.0\n2021-03-10T17:35:00+0800,log.gc_pause_max,0.09\n2021-03-10T17:35:00+0800,log.row_count,191.0\n2021-03-10T17:36:00+0800,log.gc_pause_max,0.09\n2021-03-10T17:36:00+0800,log.error_count,0.0\n2021-03-10T17:36:00+0800,log.row_count,216.0\n2021-03-10T17:37:00+0800,log.row_count,174.0\n', 'timestamp,cmdb_id,message\n2021-03-10T17:59:24+0800,Tomcat04,IG01 POST /UOCP/person/ServiceTest9.json HTTP/1.1 500 1194 - k6/0.29.0 (api_url 60011 60.011\n')```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz
import re

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Reuse existing DataFrames from kernel state: metric_df, trace_df, log_df
# Ensure timestamp tz-aware columns exist
for df, name in [(metric_df,'metric_df'), (trace_df,'trace_df'), (log_df,'log_df')]:
    if 'ts' not in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        if df['ts'].dt.tz is None:
            df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window (inclusive) in UTC+8
start = pd.Timestamp('2021-03-10 17:30:00').tz_localize(tz)
end = pd.Timestamp('2021-03-10 18:00:00').tz_localize(tz)

# Target component
cm = 'Tomcat04'

# Desired keys (attempt to find them if present)
desired = {
    'trace.self.duration_mean': ('trace', 'trace_name'),
    'trace.self.duration_p95': ('trace', 'trace_name'),
    'log.row_count': ('log', 'log_name'),
    'log.gc_pause_max': ('log', 'log_name'),
    'log.error_count': ('log', 'log_name'),
    # CPU KPI may have slightly different naming; we'll search for best match in metric_df for Tomcat04
    'OSLinux-CPU_CPU_CPUCpuUtil (candidate)': ('metric', 'kpi_name'),
    'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed': ('metric', 'kpi_name'),
    'JVM-Memory_7778_JVM_Memory_HeapMemoryUsage': ('metric', 'kpi_name'),
}

# Discover actual CPU KPI name for Tomcat04 by pattern matching in metric_df
kpis_tomcat04 = metric_df[metric_df['cmdb_id']==cm]['kpi_name'].dropna().unique().tolist()
cpu_candidates = [k for k in kpis_tomcat04 if re.search(r'cpu', k, re.IGNORECASE) and re.search(r'Util|util|CPU', k)]
# prefer exact "CPUCpuUtil" style if exists
cpu_kpi = None
if cpu_candidates:
    # try to select one with 'CPUCpuUtil' or 'CpuUtil' or 'Cpu' in name
    for cand in cpu_candidates:
        if re.search(r'CPUCpuUtil|CpuUtil|CPUUtil|Cpu', cand, re.IGNORECASE):
            cpu_kpi = cand
            break
    if cpu_kpi is None:
        cpu_kpi = cpu_candidates[0]
# Replace candidate name with discovered one if found
if cpu_kpi:
    desired.pop('OSLinux-CPU_CPU_CPUCpuUtil (candidate)')
    desired[cpu_kpi] = ('metric','kpi_name')
else:
    # remove candidate if none found
    desired.pop('OSLinux-CPU_CPU_CPUCpuUtil (candidate)')

# Prepare results list
rows = []

# Helper to compute p50 and p95 on full series for cmdb_id + name
def compute_percentiles_full(df, id_col, name_col, cm, name):
    s = df[(df[id_col]==cm) & (df[name_col]==name)]['value']
    if s.empty:
        return None
    p50 = float(np.nanpercentile(s.values,50))
    p95 = float(np.nanpercentile(s.values,95))
    return round(p50,6), round(p95,6)

# Loop through desired keys and evaluate stricter anomaly rule
for metric_name, (df_type, name_col) in desired.items():
    if df_type == 'trace':
        df = trace_df
        id_col = 'cmdb_id'
    elif df_type == 'log':
        df = log_df
        id_col = 'cmdb_id'
    else:
        df = metric_df
        id_col = 'cmdb_id'
    # find exact series name: for trace/log/metric we need the name value equal to metric_name (for matched cpu it's exact)
    # For safety, check presence
    exists = ((df[id_col]==cm) & (df[name_col]==metric_name)).any()
    if not exists:
        # skip if not present
        continue
    # compute global percentiles using full series (rule 9)
    pct = compute_percentiles_full(df, id_col, name_col, cm, metric_name)
    if pct is None:
        continue
    p50, p95 = pct
    # filter to incident window first
    window = df[(df[id_col]==cm) & (df[name_col]==metric_name) & (df['ts']>=start) & (df['ts']<=end)].copy()
    if window.empty:
        continue
    # stricter anomaly rule: value >= global_p95 AND value > global_p50 * 1.5
    window['is_anom'] = (window['value'] >= p95) & (window['value'] > (p50 * 1.5))
    anom_points = window[window['is_anom']].sort_values('ts')
    if anom_points.empty:
        continue
    earliest = anom_points['ts'].min().tz_convert(tz).strftime('%Y-%m-%dT%H:%M:%S%z')
    first_val = float(anom_points.loc[anom_points['ts']==anom_points['ts'].min(),'value'].iloc[0])
    rows.append({
        'metric_name': metric_name,
        'global_p50': p50,
        'global_p95': p95,
        'earliest_anomaly_timestamp': earliest,
        'anomaly_value': round(first_val,6)
    })

# Prepare compact output DataFrame limited to top 20 rows
anomalies_df = pd.DataFrame(rows)
if not anomalies_df.empty:
    anomalies_df = anomalies_df.sort_values(['earliest_anomaly_timestamp','metric_name']).head(20).reset_index(drop=True)

# Also return up to 20 raw trace.csv rows for Tomcat04 within the window
tomcat04_traces = trace_df[(trace_df['cmdb_id']==cm) & (trace_df['ts']>=start) & (trace_df['ts']<=end)].copy().sort_values('ts').head(20)
if not tomcat04_traces.empty:
    tomcat04_traces_out = tomcat04_traces[['ts','trace_name','value']].copy()
    tomcat04_traces_out['timestamp'] = tomcat04_traces_out['ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    tomcat04_traces_out = tomcat04_traces_out[['timestamp','trace_name','value']].reset_index(drop=True)
else:
    tomcat04_traces_out = tomcat04_traces[['ts','trace_name','value']].copy()

# Display compact results as DataFrames (anomalies summary and raw trace rows)
anomalies_df, tomcat04_traces_out

```
Out[6]:
```
Summary of Tomcat04 stricter-anomaly check (rule: value >= global_p95 AND value > 1.5 * global_p50, using full-series percentiles; window 2021-03-10 17:30–18:00 UTC+8):

Anomalous metrics (CSV-like rows: metric_name,global_p50,global_p95,earliest_anomaly_timestamp,anomaly_value)
- log.row_count,208.0,349.0,2021-03-10T17:30:00+0800,424.0
- log.error_count,0.0,0.0,2021-03-10T17:59:00+0800,1.0

Notes:
- log.row_count: a clear spike at 17:30 (424) exceeds the stricter threshold (p95=349 and > 1.5*p50). This indicates a significant request/load burst on Tomcat04 at the start of the incident window.
- log.error_count: p50 and p95 are both 0, so a non-zero error_count (1.0 at 17:59) passes the rule; this is a single error occurrence late in the window and should be inspected but may be a discrete event rather than a sustained anomaly.

Raw trace rows for Tomcat04 within the window (up to 20 rows; examples)
- 2021-03-10T17:30:00+0800, trace.self.row_count, 2813.0
- 2021-03-10T17:30:00+0800, trace.to_MG01.row_count, 204.0
- 2021-03-10T17:30:00+0800, trace.to_MG02.row_count, 215.0
- 2021-03-10T17:31:00+0800, trace.self.row_count, 2377.0
- 2021-03-10T17:30:00+0800, trace.self.duration_mean, 0.118433
(20 rows were returned in total; these show very large per-minute row_count values at 17:30–17:31 and various duration metrics for calls to MG01/MG02.)

Actionable takeaways
- Investigate the load spike at 17:30 on Tomcat04 (high log.row_count and very large trace.self.row_count). Check application logs, request patterns, upstream load balancer / apache front-end metrics, and downstream MG service capacity around 17:30.
- Inspect the single error_count = 1 at 17:59 (Tomcat04 error log) for context and whether it correlates with other anomalies (latency or resource issues).
- Correlate these findings with the earlier trace and metric analyses (tail latency on Tomcat nodes and IG/MG traces) to identify root cause (high incoming load, downstream slowness, GC/CPU, or app errors).

The original code execution output of IPython Kernel is also provided below for reference:

(       metric_name  global_p50  global_p95 earliest_anomaly_timestamp  anomaly_value
0    log.row_count       208.0       349.0   2021-03-10T17:30:00+0800          424.0
1  log.error_count         0.0         0.0   2021-03-10T17:59:00+0800            1.0,                    timestamp                     trace_name        value
0   2021-03-10T17:30:00+0800  trace.from_IG01.duration_mean     0.728512
1   2021-03-10T17:30:00+0800        trace.to_MG02.row_count   215.000000
2   2021-03-10T17:30:00+0800     trace.to_MG02.duration_p95     3.121800
3   2021-03-10T17:30:00+0800    trace.to_MG02.duration_mean     0.674740
4   2021-03-10T17:30:00+0800        trace.to_MG01.row_count   204.000000
5   2021-03-10T17:30:00+0800     trace.to_MG01.duration_p95     3.257800
6   2021-03-10T17:30:00+0800           trace.self.row_count  2813.000000
7   2021-03-10T17:30:00+0800        trace.self.duration_p95     1.030400
8   2021-03-10T17:30:00+0800       trace.self.duration_mean     0.118433
9   2021-03-10T17:30:00+0800    trace.to_MG01.duration_mean     0.732583
10  2021-03-10T17:30:00+0800   trace.from_root.duration_p95     1.004000
11  2021-03-10T17:30:00+0800  trace.from_root.duration_mean     0.146714
12  2021-03-10T17:30:00+0800      trace.from_IG02.row_count   216.000000
13  2021-03-10T17:30:00+0800   trace.from_IG02.duration_p95     3.674500
14  2021-03-10T17:30:00+0800  trace.from_IG02.duration_mean     0.874366
15  2021-03-10T17:30:00+0800      trace.from_IG01.row_count   201.000000
16  2021-03-10T17:30:00+0800   trace.from_IG01.duration_p95     3.123000
17  2021-03-10T17:30:00+0800      trace.from_root.row_count    14.000000
18  2021-03-10T17:31:00+0800           trace.self.row_count  2377.000000
19  2021-03-10T17:31:00+0800        trace.to_MG02.row_count   185.000000)```
```